# Week 2: Provider-Agnostic LLM Client (Gemini + Ollama/Llama 3.2)

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (CPU also works, just slower)
2. Add a Colab **Secret** (key icon, left sidebar): name `GEMINI_API_KEY`, paste your key from aistudio.google.com/apikey, and turn on **Notebook access**

Run all cells in order. If the runtime restarts, re-run everything from the top.

## Step 1: Project folder

In [1]:
import os
os.makedirs("/content/week-2", exist_ok=True)
%cd /content/week-2
!pwd

/content/week-2
/content/week-2


## Step 2: Install Ollama

In [2]:
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Step 3: Start the Ollama server in the background

In [3]:
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://localhost:11434")
        if r.ok:
            print("Server says:", r.text)
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print("Server did not start. Re-run this cell.")

Server says: Ollama is running


## Step 4: Download Llama 3.2 (~2 GB)

In [4]:
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


## Step 5: Raw test of the local model

In [5]:
!curl -s http://localhost:11434/api/generate -d '{"model": "llama3.2", "prompt": "Say hello in one short sentence.", "stream": false}'

{"model":"llama3.2","created_at":"2026-11-23T06:48:04.026730604Z","response":"Hello.","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,2875,11914,13,128009,128006,78191,128007,271,9906,13],"total_duration":107804126982,"load_duration":56563277244,"prompt_eval_count":32,"prompt_eval_cached_count":0,"prompt_eval_duration":51199208000,"eval_count":3,"eval_duration":38695000}

## Step 6: Create `.env` from the Colab secret

In [6]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

with open("/content/week-2/.env", "w") as f:
    f.write(f'GEMINI_API_KEY="{key}"\n')

print(".env created. Key length:", len(key))

.env created. Key length: 53


## Step 7: requirements.txt and install

In [7]:
%%writefile requirements.txt
python-dotenv
pytest
google-genai
requests

Writing requirements.txt


In [8]:
!pip install -q -r requirements.txt
from google import genai
print("google-genai imported OK")

google-genai imported OK


## Step 8: `llm_client.py`

In [9]:
%%writefile llm_client.py
from abc import ABC, abstractmethod
from google import genai
import requests


# Common interface: every provider must implement generate()
class LLMProvider(ABC):

    @abstractmethod
    def generate(self, prompt):
        pass


# Gemini cloud provider
class GeminiProvider(LLMProvider):

    FALLBACK_MODELS = ["gemini-3.5-flash", "gemini-2.5-flash"]

    def __init__(self, api_key, model="gemini-3.6-flash"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def generate(self, prompt):
        models_to_try = [self.model] + [
            m for m in self.FALLBACK_MODELS if m != self.model
        ]
        last_error = None
        for model in models_to_try:
            try:
                response = self.client.models.generate_content(
                    model=model,
                    contents=prompt
                )
                self.model = model  # remember the one that worked
                return response.text
            except Exception as e:
                last_error = e
        raise RuntimeError(f"All Gemini models failed. Last error: {last_error}")


# Ollama local provider
class OllamaProvider(LLMProvider):

    def __init__(self, model="llama3.2"):
        self.model = model
        self.url = "http://localhost:11434/api/generate"

    def generate(self, prompt):
        response = requests.post(
            self.url,
            json={
                "model": self.model,
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )
        response.raise_for_status()
        return response.json()["response"]


# Common client: the rest of the program only talks to this class
class LLMClient:

    def __init__(self, provider):
        self.provider = provider

    def generate(self, prompt):
        return self.provider.generate(prompt)

Writing llm_client.py


## Step 9: `main.py`

In [10]:
%%writefile main.py
import os
from dotenv import load_dotenv
from llm_client import (
    GeminiProvider,
    OllamaProvider,
    LLMClient
)

load_dotenv(override=True)

provider_name = input(
    "Choose provider (gemini/ollama): "
).strip().lower()

if provider_name == "gemini":
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Gemini API key not found in .env")
    provider = GeminiProvider(api_key)

elif provider_name == "ollama":
    provider = OllamaProvider("llama3.2")

else:
    raise ValueError("Invalid provider selected")

client = LLMClient(provider)

prompt = input("Enter your question: ")
answer = client.generate(prompt)

print("\nLLM Response:")
print(answer)

Writing main.py


## Step 10: Unit test with a fake provider

In [11]:
%%writefile test_client.py
import unittest
from llm_client import LLMClient


# Fake provider for testing
class FakeProvider:
    def generate(self, prompt):
        return "AI stands for Artificial Intelligence."


class TestLLMClient(unittest.TestCase):

    def test_generate(self):
        provider = FakeProvider()
        client = LLMClient(provider)
        result = client.generate("Explain AI")
        self.assertEqual(
            result,
            "AI stands for Artificial Intelligence."
        )


if __name__ == "__main__":
    unittest.main()

Writing test_client.py


In [12]:
!python -m pytest test_client.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week-2
plugins: langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
collected 1 item                                                               

test_client.py::TestLLMClient::test_generate PASSED                      [100%]

============================== 1 passed in 1.72s ===============================


## Step 11: Run the program

Use `%run` (not `!python`) so `input()` prompts work. Run once and type `ollama`, then run again and type `gemini`.

In [13]:
%run main.py

Choose provider (gemini/ollama): ollama
Enter your question: Explain AI in 2 lines

LLM Response:
Artificial intelligence (AI) refers to the simulation of human intelligence in machines that can think, learn, and act autonomously. AI systems use algorithms and data to make decisions, solve problems, and perform tasks, often with remarkable accuracy and speed.


In [14]:
%run main.py

Choose provider (gemini/ollama): gemini
Enter your question: Explain AI in 2 lines

LLM Response:
Artificial Intelligence (AI) is technology that enables computers to mimic human intelligence by learning from data, reasoning, and making decisions. 
It allows machines to solve complex problems, understand language, and perform tasks autonomously that previously required human thought.
